# NB05 — Cross-cohort meta-analysis, the H2 control-type analysis, and the synaptic gene set S
**Meta-analysis:** DerSimonian-Laird random-effects pooling of per-cohort log2 fold changes over the **shared gene
universe** (genes tested in every cohort). Reports Q, tau2, I2, direction consistency, BH-FDR, leave-one-cohort-out,
and mutually exclusive gene classes. With 3 cohorts, I2 and random-effects p-values are descriptive (NB13 adds a
rank-based check).

**H2 (control-type sensitivity):** pairwise effect-size concordance between control types (all genes, S genes, other
genes), direction of S genes per control type, and enrichment of non-malignant brain-cell markers among DEGs.
Pre-specified support: Spearman rho < 0.7 for S genes in any control pair, or marker enrichment above chance.

**Also here:** MSigDB export for NB06 and the synaptic set **S**. S is taken from your SynGO files
(`03_Processed_Datasets/SynGO/syngo_genes.csv`, made in NB01, or `02_Raw_Datasets/SynGO/genes.xlsx`).
GO CC "synapse" is kept as an alternative set for NB13.

In [ ]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
M = CFG["meta"]; DEc = CFG["de"]; RAW = P("raw")
print("SMOKE =", SMOKE)

In [ ]:
# ---- MSigDB export (used by NB06) ----------------------------------------------------------------------
R_SETS = r'''
suppressMessages(library(msigdbr))
get_sets <- function(coll, sub = NA) {
  a1 <- list(species = "Homo sapiens", collection = coll); if (!is.na(sub)) a1$subcollection <- sub
  r <- tryCatch(suppressWarnings(do.call(msigdbr, a1)), error = function(e) NULL)
  if (is.null(r) || nrow(r) == 0) {
    a2 <- list(species = "Homo sapiens", category = coll); if (!is.na(sub)) a2$subcategory <- sub
    r <- tryCatch(suppressWarnings(do.call(msigdbr, a2)), error = function(e) NULL)
  }
  if (is.null(r) || nrow(r) == 0) stop(paste("msigdbr failed for", coll, sub, "- install msigdbdf (see NB00)"))
  data.frame(collection = ifelse(is.na(sub), coll, sub), gs_name = r$gs_name, gene_symbol = r$gene_symbol)
}
all <- rbind(get_sets("H"), get_sets("C5", "GO:BP"), get_sets("C5", "GO:CC"), get_sets("C5", "GO:MF"),
             get_sets("C2", "CP:REACTOME"))
all <- unique(all)
write.csv(all, commandArgs(TRUE)[1], row.names = FALSE)
cat("gene-set rows:", nrow(all), "| sets:", length(unique(all$gs_name)), "| msigdbr", as.character(packageVersion("msigdbr")), "\n")
'''
sets_csv = P("enrich", "msigdb_long.csv.gz")
if not sets_csv.exists():
    tmp = P("enrich", "msigdb_long.csv")
    print(run_r(write_r("export_genesets.R", R_SETS), [tmp]))
    pd.read_csv(tmp).to_csv(sets_csv, index=False); tmp.unlink()
gsets = pd.read_csv(sets_csv)
print(gsets.groupby("collection").gs_name.nunique().to_string())

In [ ]:
# ---- synaptic set S (SynGO) -------------------------------------------------------------------------------
proc = Path(str(ROOT)) / "03_Processed_Datasets" / "SynGO" / "syngo_genes.csv"
xlsx = RAW / "SynGO" / "genes.xlsx"
if proc.exists():
    sy = pd.read_csv(proc); S_file = proc
elif xlsx.exists():
    sy = pd.read_excel(xlsx, engine="openpyxl"); S_file = xlsx
else:
    sy, S_file = None, None
S_alt = sorted(set(gsets.loc[gsets.gs_name == "GOCC_SYNAPSE", "gene_symbol"].astype(str)))
if sy is not None:
    col = find_col(sy, ["hgnc_symbol", "gene_symbol", "symbol"])
    S_main = sorted(set(sy[col].dropna().astype(str).str.strip()) - {""})
    S_source = f"SynGO genes ({S_file.name}; sha256 {sha256(S_file)[:12]})"
else:
    S_main, S_source = S_alt, "GOCC_SYNAPSE (FALLBACK: SynGO files not found; broader set)"
    log("SynGO not found; using GO CC synapse as S. Record this in the manuscript.", "WARN")
require(len(S_main) >= 100, f"Synaptic set unexpectedly small ({len(S_main)} genes).")
pd.DataFrame({"gene": S_main, "source": S_source}).to_csv(P("final", "S_candidate_genes.csv"), index=False)
pd.DataFrame({"gene": S_alt, "source": "GOCC_SYNAPSE"}).to_csv(P("final", "S_alt_GOCC_SYNAPSE.csv"), index=False)
manifest_update("gene_sets", "S", {"source": S_source, "n_genes": len(S_main), "n_alt_gocc_synapse": len(S_alt),
                                   "overlap_syngo_gocc": len(set(S_main) & set(S_alt))})
print("S:", S_source, "|", len(S_main), "genes | GO CC synapse:", len(S_alt), "| overlap:", len(set(S_main) & set(S_alt)))
frozen_note = P("config", "frozen_definitions.json")
if frozen_note.exists():
    print("NOTE: S/M/R are already frozen (NB10). S_candidate_genes.csv must stay identical; verify_frozen() checks this.")
    verify_frozen()

In [ ]:
# ---- per-cohort DE tables and random-effects meta-analysis ---------------------------------------------------
ORDER = ["GSE4290", "GSE50161", "TOIL_TCGA_vs_GTEx"]
studies = {}
for f in sorted(P("de").glob("*_DE_full.csv"), key=lambda p: ORDER.index(p.name[:-12]) if p.name[:-12] in ORDER else 99):
    d = pd.read_csv(f).dropna(subset=["logFC", "SE"])
    d = d[d.SE > 0].drop_duplicates("gene").set_index("gene")
    studies[f.name.replace("_DE_full.csv", "")] = d
require(len(studies) >= M["min_studies"], f"Meta-analysis needs at least {M['min_studies']} cohorts; found {list(studies)}. Run NB04.")
universe = sorted(set.intersection(*[set(d.index) for d in studies.values()]))
print({k: len(v) for k, v in studies.items()}, "| shared gene universe:", len(universe))
require(len(universe) > 5000, "Shared gene universe < 5000; gene IDs are probably not harmonised (check NB04 output).")
manifest_update("meta_analysis", "shared_universe_size", len(universe))
names = list(studies)
Y = np.column_stack([studies[k].loc[universe, "logFC"].values for k in names])
V = np.column_stack([studies[k].loc[universe, "SE"].values ** 2 for k in names])
Pn = np.column_stack([studies[k].loc[universe, "adj.P.Val"].values for k in names])
Pr = np.column_stack([studies[k].loc[universe, "P.Value"].values for k in names])

full = dl_meta(Y, V)
res = pd.DataFrame({"gene": universe, "meta_log2FC": full["mu"], "meta_SE": full["se"], "meta_p": full["p"],
                    "Q": full["Q"], "tau2": full["tau2"], "I2": full["I2"]})
res["meta_FDR"] = multipletests(res.meta_p, method="fdr_bh")[1]
for i, k in enumerate(names):
    res[f"logFC_{k}"] = Y[:, i]; res[f"adjP_{k}"] = Pn[:, i]
sig_each = Pn < DEc["fdr"]; sign = np.sign(Y)
res["n_cohorts_sig"] = sig_each.sum(1); res["direction_consistent"] = (sign == sign[:, [0]]).all(1)

for j, k in enumerate(names):                     # leave-one-cohort-out + replication in the held-out cohort
    keep = [i for i in range(Y.shape[1]) if i != j]
    loo = dl_meta(Y[:, keep], V[:, keep])
    fdr_loo = multipletests(loo["p"], method="fdr_bh")[1]
    replicates = (np.sign(Y[:, j]) == np.sign(loo["mu"])) & (Pr[:, j] < 0.05)
    res[f"LOO_FDR_without_{k}"] = fdr_loo
    res[f"fails_holdout_{k}"] = (fdr_loo < M["fdr"]) & ~replicates
res["n_failed_holdouts"] = res[[c for c in res.columns if c.startswith("fails_holdout_")]].sum(axis=1)


def classify(r):
    if r.n_cohorts_sig == len(names) and r.direction_consistent:
        return "consistently_significant"
    if r.meta_FDR < M["fdr"] and r.I2 < M["i2_heterogeneous"] and r.n_failed_holdouts == 0:
        return "meta_significant"
    if r.meta_FDR < M["fdr"] and r.n_failed_holdouts > 0:
        return "failed_independent_validation"
    if r.meta_FDR < M["fdr"]:
        return "heterogeneous"
    if r.n_cohorts_sig >= 1:
        return "cohort_specific"
    return "not_significant"


res["class"] = res.apply(classify, axis=1)
res["is_S"] = res.gene.isin(set(S_main))
res.to_csv(P("meta_analysis", "meta_analysis_full.csv"), index=False)
print(res["class"].value_counts().to_string())
print("\nClass counts among S genes:\n", res[res.is_S]["class"].value_counts().to_string())
manifest_update("meta_analysis", "class_counts", res["class"].value_counts().to_dict())
manifest_update("meta_analysis", "studies", names)

In [ ]:
# ---- H2: control-type concordance, S direction, marker enrichment -------------------------------------------
rows = []
for a in range(len(names)):
    for b in range(a + 1, len(names)):
        for label, sub in [("all genes", res), ("S genes", res[res.is_S]), ("non-S genes", res[~res.is_S])]:
            rho, p = stats.spearmanr(sub[f"logFC_{names[a]}"], sub[f"logFC_{names[b]}"])
            rows.append({"pair": f"{names[a]} vs {names[b]}", "gene_set": label, "n": len(sub), "spearman_rho": rho, "p": p})
conc = pd.DataFrame(rows); conc.to_csv(P("meta_analysis", "H2_control_type_concordance.csv"), index=False)
print(conc.round(4).to_string(index=False))

dirrows = []
for k in names:
    s = res[res.is_S][f"logFC_{k}"]; sig = res[res.is_S][f"adjP_{k}"] < DEc["fdr"]
    dirrows.append({"cohort": k, "n_S_genes": len(s), "median_log2FC_S": s.median(), "fraction_S_down_in_GBM": float((s < 0).mean()),
                    "n_S_sig_down": int(((s < 0) & sig).sum()), "n_S_sig_up": int(((s > 0) & sig).sum())})
pd.DataFrame(dirrows).to_csv(P("meta_analysis", "H2_S_direction_by_control.csv"), index=False)
print(pd.DataFrame(dirrows).round(3).to_string(index=False))

mk = CFG["markers"]; mrows = []
for k in names:
    sig_genes = set(res.loc[(res[f"adjP_{k}"] < DEc["fdr"]) & (res[f"logFC_{k}"].abs() >= DEc["min_abs_log2fc"]), "gene"])
    for cat in ["neuron_identity", "oligodendrocyte", "astrocyte_normal", "immune_stromal"]:
        marks = set(mk[cat]) & set(universe)
        a = len(sig_genes & marks); b = len(marks) - a
        c = len(sig_genes) - a; d_ = len(universe) - len(sig_genes) - b
        odds, p = stats.fisher_exact([[a, b], [c, d_]], alternative="greater")
        mrows.append({"cohort": k, "marker_set": cat, "markers_in_universe": len(marks), "markers_significant": a,
                      "n_sig_genes": len(sig_genes), "expected": len(marks) * len(sig_genes) / len(universe),
                      "odds_ratio": odds, "fisher_p": p})
mk_df = pd.DataFrame(mrows)
mk_df["fisher_FDR"] = multipletests(mk_df.fisher_p, method="fdr_bh")[1]
mk_df.to_csv(P("meta_analysis", "H2_marker_enrichment.csv"), index=False)
print(mk_df.round(4).to_string(index=False))
manifest_update("H2", "note", "marker lists are small curated sets; one-sided Fisher test, BH across cohort x marker set")

In [ ]:
# ---- figures -------------------------------------------------------------------------------------------
pairs = [(names[a], names[b]) for a in range(len(names)) for b in range(a + 1, len(names))]
fig, axes = plt.subplots(len(pairs), 2, figsize=(9, 4.0 * len(pairs)), squeeze=False)
for row, (a, b) in zip(axes, pairs):
    for axi, (label, sub, c) in zip(row, [("non-S genes", res[~res.is_S], "#b8bec9"), ("S (synaptic) genes", res[res.is_S], "#1f3a93")]):
        axi.scatter(sub[f"logFC_{a}"], sub[f"logFC_{b}"], s=3, alpha=.4, c=c, rasterized=True)
        rho = stats.spearmanr(sub[f"logFC_{a}"], sub[f"logFC_{b}"])[0]
        axi.axhline(0, c="k", lw=.4); axi.axvline(0, c="k", lw=.4)
        axi.set_title(f"{label} (rho={rho:.2f})", fontsize=10); axi.set_xlabel(f"log2FC {a}"); axi.set_ylabel(f"log2FC {b}")
fig.tight_layout(); savefig(fig, "H2_logFC_concordance", folder="meta_analysis"); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.4, 4.6))
sigm = res.meta_FDR < M["fdr"]
lp = -np.log10(res.meta_p.clip(lower=1e-300))
ax.scatter(res.meta_log2FC[~sigm], lp[~sigm], s=4, c="#b8bec9", rasterized=True)
ax.scatter(res.meta_log2FC[sigm], lp[sigm], s=5, c="#1f3a93", rasterized=True)
ax.set_xlabel("meta log2FC (GBM vs control)"); ax.set_ylabel("-log10 meta p"); ax.set_title("Random-effects meta-analysis")
savefig(fig, "Meta_volcano", folder="meta_analysis"); plt.close(fig)
print("\nNB05 complete. Next: NB06 (functional enrichment).")